# 오목 AlphaZero 에이전트 (Gomoku, 렌주룰)

15×15 오목을 **ResNet + MCTS 자기대전(AlphaZero 방식)**으로 학습합니다. Google Colab 무료 GPU(T4)에서 돌아가도록 만들었습니다.

```
① 자기대전 → ② 리플레이 버퍼 → ③ 학습 모델 훈련 → ④ 아레나 대결 → ⑤ 교체 여부 결정 → ①
```

**규칙: 렌주룰 (1번 셀 `CFG['rule']`에서 변경 가능)**
- **흑만 금수**: 3-3, 4-4, 6목 이상 (금수 자리에는 둘 수 없음)
- **백은 제한 없음**, 백은 6목 이상도 승리
- 정확히 5목이 되는 수는 금수보다 우선 (흑도 5목이면 승리)
- 흑 첫 수는 천원(정중앙). 대회용 개시 규칙(스왑 등)은 생략
- `rule='korean'`으로 바꾸면 한국식(흑백 모두 3-3 금지), 체크포인트는 규칙마다 다른 폴더에 저장

**아발론 프로젝트에서 배운 것을 처음부터 반영**
- 자기대전의 40%는 휴리스틱 봇과 대국 (소극적 플레이·무승부 수렴 방지), 연습판은 초반 무작위 2수·잡음 없음 (실전처럼)
- 학습에 쓰지 않는 **평가 전용 미니맥스 봇**으로 실력 측정
- **이길 수 있으면 바로 두고, 상대 5목 자리는 바로 막기** (`tactical_override`) — 마무리 못 하는 문제 예방
- 8가지 회전·반사 대칭 증강, 계단식 학습률
- **판 길이를 반영한 가치 목표**: 빨리 이길수록 좋고, 질 때도 오래 버틸수록 덜 나쁘게 (`len_shaping`) — 한쪽이 짧은 수순으로만 이기는 쏠림 방지

**실행 방법**
1. `런타임 → 런타임 유형 변경 → T4 GPU`
2. `모두 실행` → 구글 드라이브 연결 허용
3. 체크포인트는 `MyDrive/gomoku_renju_az/`에 반복마다 저장 — 세션이 끊기면 다시 `모두 실행`하면 이어서 학습
4. 학습 후 AI와 두기: 1, 2, 4, 5, 6, 7, 8번 셀 실행 → **12번(클릭)** 또는 11번(좌표 입력) 셀


In [ ]:
# ===== 1. 환경 설정 & 하이퍼파라미터 =====
import os, sys, time, json, math, random
import numpy as np

try:
    import numba
except ImportError:
    !pip -q install numba
    import numba
from numba import njit

import torch
import torch.nn as nn
import torch.nn.functional as F

# 코랩 호스팅 런타임이면 구글 드라이브에, 로컬 런타임(친구 PC 등)이나 Jupyter면 현재 폴더에 저장
SAVE_ROOT = '.'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_ROOT = '/content/drive/MyDrive'
except Exception:
    print('구글 드라이브 연결 안 함 → 로컬 폴더에 저장합니다')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

CFG = dict(
    # --- 규칙 ---
    board_size=15,
    rule='renju',            # 'renju' = 렌주룰(흑만 3-3·4-4·6목 금수, 백은 6목도 승리)
                             # 'korean' = 한국식(흑백 모두 3-3 금지), 'black' = 흑만 3-3 금지, 'none' = 금지 없음
    exact_five=True,         # renju 외 규칙에서: True = 6목 이상은 승리 아님, False = 5목 이상이면 승리
    black_first_center=True, # 흑 첫 수는 천원(정중앙)
    # --- 신경망 ---
    channels=64, blocks=6,
    # --- 자기대전 ---
    iterations=200,
    games_per_iter=64,
    sims=200,                # 수당 MCTS 탐색 횟수 (128 → 200: 4-3·금수 유도 같은 위협을 더 깊이 읽도록)
    c_puct=1.5,
    dirichlet_alpha=0.1, dirichlet_eps=0.25,
    temp_moves=6,            # 처음 몇 수는 방문 비율대로 무작위 선택(다양성), 이후 최다 방문
    tactical_override=True,  # 이길 수 있으면 바로 두고, 상대 5목 자리는 바로 막기
    heuristic_frac=0.4,      # 자기대전 중 휴리스틱 봇과 두는 판 비율 (AI 쪽 국면만 학습, 0.25 → 0.4: 자기 약점만 찌르는 수순 방지)
    heuristic_temp_moves=2,  # 휴리스틱 판에서 AI 초반 무작위 수 (실전처럼 짧게)
    heuristic_noise=False,   # 휴리스틱 판에서는 탐색 잡음 끔
    # --- 가치 목표 ---
    len_shaping=0.5,         # 판 길이 반영: 승패 크기 = 1 - len_shaping × min(수, len_scale) / len_scale
    len_scale=100,           #   → 빨리 이길수록 +1에 가깝고, 오래 버티다 질수록 -1에서 멀어짐 (0이면 끔)
    # --- 학습 ---
    buffer_size=150_000,
    batch_size=256,
    train_steps=400,
    lr_schedule={0: 1e-3, 75: 3e-4, 150: 1e-4},
    weight_decay=1e-4,
    # --- 아레나 / 평가 ---
    arena_games=20, arena_threshold=0.55, arena_temp_moves=4,
    eval_every=3, eval_games=20,   # 휴리스틱·미니맥스 상대 평가 (미니맥스는 평가 전용)
    play_sims=400,                 # 사람과 둘 때 탐색 횟수
)
N = CFG['board_size']
NN = N * N
RULE_MODE = {'none': 0, 'black': 1, 'korean': 2, 'both': 2, 'renju': 3}[CFG['rule']]
EXACT = CFG['exact_five']

def EX(color):
    """color 쪽의 5목 판정 방식: True = 정확히 5목만 승리 (렌주: 흑만 정확히 5, 백은 6목 이상도 승리)"""
    return (color == 1) if RULE_MODE == 3 else EXACT

# 규칙마다 체크포인트 폴더를 따로 씀 (한국식은 예전 폴더 gomoku_az 그대로)
_folder = 'gomoku_az' if RULE_MODE == 2 else f"gomoku_{CFG['rule']}_az"
SAVE_DIR = os.path.join(SAVE_ROOT, _folder)
os.makedirs(SAVE_DIR, exist_ok=True)
RULE_NAME = {0: '금지 없음', 1: '흑만 3-3 금지', 2: '한국식 (흑백 모두 3-3 금지)', 3: '렌주 (흑만 3-3·4-4·6목 금수)'}[RULE_MODE]
print(f'{N}x{N} 오목 | 규칙: {RULE_NAME} | 저장 위치: {SAVE_DIR}')


## 2. 오목 규칙
렌주 흑 금수 판정
- **6목 이상**: 한 줄에 흑돌 6개 이상이 이어짐
- **4-4**: 한 수 더 두면 5목이 되는 "4"가 두 개 이상 (한 줄 안의 ●_●●●_● 포함)
- **3-3**: 한 수 더 두면 열린 4(_●●●●_)가 되는 "열린 3"이 두 방향 이상
- 정확히 5목이 되면 금수보다 우선

3-3에서 열린 4로 가는 자리가 다시 금수인지까지 따지는 정식 재귀 판정은 생략했습니다.

In [ ]:
# ===== 2. 오목 규칙 (numba 가속) =====
# 돌: 흑 = 1, 백 = -1, 빈칸 = 0
# 한 줄(line) 표현: 중심 ±5칸, 11칸 배열. 1 = 내 돌, 0 = 빈칸, 2 = 상대 돌 또는 판 밖
DR = np.array([0, 1, 1, 1], dtype=np.int64)
DC = np.array([1, 0, 1, -1], dtype=np.int64)

@njit
def _count(board, r, c, dr, dc, color, n):
    k = 0
    rr, cc = r + dr, c + dc
    while 0 <= rr < n and 0 <= cc < n and board[rr, cc] == color:
        k += 1; rr += dr; cc += dc
    return k

@njit
def makes_five(board, r, c, color, n, exact):
    """(r,c)에 color 돌이 놓여 있다고 보고 5목 완성 여부"""
    for d in range(4):
        k = 1 + _count(board, r, c, DR[d], DC[d], color, n) + _count(board, r, c, -DR[d], -DC[d], color, n)
        if exact:
            if k == 5: return True
        elif k >= 5:
            return True
    return False

@njit
def get_line(board, r, c, d, color, n):
    line = np.empty(11, dtype=np.int8)
    for i in range(11):
        rr = r + (i - 5) * DR[d]; cc = c + (i - 5) * DC[d]
        if 0 <= rr < n and 0 <= cc < n:
            v = board[rr, cc]
            line[i] = 1 if v == color else (0 if v == 0 else 2)
        else:
            line[i] = 2
    return line

@njit
def line_five_with_center(line, exact):
    """중심(5번)을 포함한 5목이 줄에 있는지"""
    for s in range(1, 6):          # 5칸 창 시작 위치 s..s+4, 중심 포함
        ok = True
        for i in range(s, s + 5):
            if line[i] != 1: ok = False; break
        if ok:
            if not exact: return True
            left = line[s - 1] if s - 1 >= 0 else 2
            right = line[s + 5] if s + 5 <= 10 else 2
            if left != 1 and right != 1: return True
    return False

@njit
def line_is_four(line, exact):
    """한 수만 더 두면 (중심 포함) 5목이 되는 '4'인지"""
    for j in range(1, 10):
        if line[j] == 0:
            line[j] = 1
            f = line_five_with_center(line, exact)
            line[j] = 0
            if f: return True
    return False

@njit
def line_open_four(line, exact):
    """중심을 포함한 열린 4 (_XXXX_): 양쪽 어디로 두어도 5목"""
    for s in range(0, 5):          # 6칸 창 s..s+5
        if line[s] == 0 and line[s + 5] == 0 and s + 1 <= 5 <= s + 4:
            ok = True
            for i in range(s + 1, s + 5):
                if line[i] != 1: ok = False; break
            if ok:
                if exact:
                    left = line[s - 1] if s - 1 >= 0 else 2
                    right = line[s + 6] if s + 6 <= 10 else 2
                    if left == 1 or right == 1: continue   # 두면 6목이 되는 자리는 열린 4가 아님
                return True
    return False

@njit
def line_open_three(line, exact):
    """한 수 더 두면 중심을 포함한 열린 4가 되는 '열린 3' (이미 4인 줄은 제외)"""
    if line_is_four(line, exact): return False
    for j in range(1, 10):
        if line[j] == 0:
            line[j] = 1
            f = line_open_four(line, exact)
            line[j] = 0
            if f: return True
    return False

@njit
def is_double_three(board, r, c, color, n, exact):
    """(r,c)에 두면 열린 3이 두 방향 이상 생기는지 (5목이 되는 수는 허용)"""
    if board[r, c] != 0: return False
    # 빠른 걸러내기: 네 방향 4칸 안에 내 돌이 4개 미만이면 3-3 불가능
    cnt = 0
    for d in range(4):
        for s in (-1, 1):
            for k in range(1, 5):
                rr = r + s * k * DR[d]; cc = c + s * k * DC[d]
                if 0 <= rr < n and 0 <= cc < n and board[rr, cc] == color: cnt += 1
    if cnt < 4: return False
    board[r, c] = color
    res = False
    if not makes_five(board, r, c, color, n, exact):
        threes = 0
        for d in range(4):
            if line_open_three(get_line(board, r, c, d, color, n), exact):
                threes += 1
        res = threes >= 2
    board[r, c] = 0
    return res

@njit
def line_four_count(line):
    """(흑, 정확히 5목 기준) 이 줄에서 중심을 포함한 '4'의 개수.
    5목 자리가 2곳 이상인데 열린 4(_XXXX_, 두 자리 간격 5)가 아니면 한 줄 안의 4-4로 보고 2개"""
    first = -1; cnt = 0; second = -1
    for j in range(1, 10):
        if line[j] == 0:
            line[j] = 1
            f = line_five_with_center(line, True)
            line[j] = 0
            if f:
                cnt += 1
                if first < 0: first = j
                elif second < 0: second = j
    if cnt == 0: return 0
    if cnt == 2 and second - first == 5: return 1
    return 1 if cnt == 1 else 2

@njit
def is_renju_forbidden(board, r, c, n):
    """렌주룰 흑 금수: 6목 이상, 4-4, 3-3 (정확히 5목이 되는 수는 금수보다 우선해서 허용)"""
    if board[r, c] != 0: return False
    cnt = 0                                  # 빠른 걸러내기: 주변에 흑돌이 적으면 금수 불가능
    for d in range(4):
        for s in (-1, 1):
            for k in range(1, 5):
                rr = r + s * k * DR[d]; cc = c + s * k * DC[d]
                if 0 <= rr < n and 0 <= cc < n and board[rr, cc] == 1: cnt += 1
    if cnt < 4: return False
    board[r, c] = 1
    res = False
    if not makes_five(board, r, c, 1, n, True):
        for d in range(4):                   # 6목 이상
            k = 1 + _count(board, r, c, DR[d], DC[d], 1, n) + _count(board, r, c, -DR[d], -DC[d], 1, n)
            if k >= 6: res = True
        if not res:                          # 4-4
            fours = 0
            for d in range(4):
                fours += line_four_count(get_line(board, r, c, d, 1, n))
            res = fours >= 2
        if not res:                          # 3-3
            threes = 0
            for d in range(4):
                if line_open_three(get_line(board, r, c, d, 1, n), True):
                    threes += 1
            res = threes >= 2
    board[r, c] = 0
    return res

@njit
def is_forbidden(board, r, c, color, n, rule_mode, exact):
    """rule_mode: 0 금지 없음, 1 흑만 3-3, 2 흑백 3-3 (한국식), 3 렌주"""
    if rule_mode == 3:
        return color == 1 and is_renju_forbidden(board, r, c, n)
    if rule_mode == 2 or (rule_mode == 1 and color == 1):
        return is_double_three(board, r, c, color, n, exact)
    return False

@njit
def legal_mask(board, color, n, rule_mode, exact):
    mask = np.zeros(n * n, dtype=np.bool_)
    for r in range(n):
        for c in range(n):
            if board[r, c] == 0 and not is_forbidden(board, r, c, color, n, rule_mode, exact):
                mask[r * n + c] = True
    return mask

@njit
def winning_moves(board, color, n, exact):
    """두면 바로 5목이 되는 빈칸 목록 (5목은 금수보다 우선). exact는 그 색의 판정 방식 EX(color)"""
    out = []
    for r in range(n):
        for c in range(n):
            if board[r, c] == 0:
                board[r, c] = color
                if makes_five(board, r, c, color, n, exact): out.append(r * n + c)
                board[r, c] = 0
    return out


class Gomoku:
    """판 상태. player = 이번에 둘 차례 (1 흑, -1 백)"""
    __slots__ = ('board', 'player', 'last', 'moves', 'winner', '_legal')

    def __init__(self):
        self.board = np.zeros((N, N), dtype=np.int8)
        self.player = 1
        self.last = -1
        self.moves = 0
        self.winner = None      # None 진행 중, 1 흑승, -1 백승, 0 무승부
        self._legal = None

    def copy(self):
        g = Gomoku.__new__(Gomoku)
        g.board = self.board.copy(); g.player = self.player; g.last = self.last
        g.moves = self.moves; g.winner = self.winner; g._legal = self._legal
        return g

    def legal(self):
        if self._legal is None:
            if self.moves == 0 and CFG['black_first_center']:
                m = np.zeros(NN, dtype=np.bool_); m[(N // 2) * N + N // 2] = True
                self._legal = m
            else:
                self._legal = legal_mask(self.board, self.player, N, RULE_MODE, EX(self.player))
                if self.winner is None and not self._legal.any():
                    self.winner = 0   # 남은 빈칸이 전부 금수면 무승부
        return self._legal

    def play(self, a):
        r, c = divmod(int(a), N)
        assert self.board[r, c] == 0, '이미 돌이 있는 자리'
        self.board[r, c] = self.player
        self.last = int(a); self.moves += 1; self._legal = None
        if makes_five(self.board, r, c, self.player, N, EX(self.player)):
            self.winner = self.player
        self.player = -self.player
        if self.winner is None and self.moves == NN:
            self.winner = 0       # 판이 꽉 차면 무승부
        return self

    def done(self):
        return self.winner is not None


COLS = 'ABCDEFGHJKLMNOPQRST'[:N]   # I 제외 (바둑판 관례)
def move_name(a):
    r, c = divmod(int(a), N)
    return f'{COLS[c]}{N - r}'

def parse_move(s):
    s = s.strip().upper()
    c = COLS.index(s[0]); r = N - int(s[1:])
    if not (0 <= r < N): raise ValueError
    return r * N + c

def show(g, mark_last=True):
    sym = {1: '●', -1: '○', 0: '·'}
    print('   ' + ' '.join(COLS))
    for r in range(N):
        row = []
        for c in range(N):
            ch = sym[int(g.board[r, c])]
            if mark_last and g.last == r * N + c: ch = '◆' if g.board[r, c] == 1 else '◇'
            row.append(ch)
        print(f'{N - r:2d} ' + ' '.join(row))

_ = Gomoku().play(NN // 2).legal()   # numba 컴파일
print('규칙 준비 완료')


## 3. 규칙 검증
3-3, 띈 3, 막힌 3, 4-3, 가장자리, 5목 우선, 4-4, 한 줄 4-4, 6목(흑 금수 / 백 승리)을 확인합니다.

In [ ]:
# ===== 3. 규칙 검증 =====
def _board_from(stones):
    """stones: {'H8': 1, ...} → Gomoku (차례는 흑)"""
    g = Gomoku()
    for name, col in stones.items():
        r, c = divmod(parse_move(name), N)
        g.board[r, c] = col
    g.moves = len(stones); g._legal = None
    return g

def _check(name, cond):
    print(('✅' if cond else '❌'), name)
    assert cond, name

def _forbidden(g, mv, color=1):
    r, c = divmod(parse_move(mv), N)
    return is_forbidden(g.board, r, c, color, N, RULE_MODE, EX(color))

def _five_after(stones, mv, color):
    g = _board_from(stones)
    r, c = divmod(parse_move(mv), N); g.board[r, c] = color
    return makes_five(g.board, r, c, color, N, EX(color))

if N == 15:
    B33 = {'F8': 1, 'G8': 1, 'H6': 1, 'H7': 1}          # H8에 두면 가로 _●●● + 세로 ●●● = 3-3
    if RULE_MODE >= 1:
        _check('흑 3-3 금지 (가로+세로)', _forbidden(_board_from(B33), 'H8'))
        _check('띈 3 포함 3-3 금지', _forbidden(_board_from({'E8': 1, 'G8': 1, 'H6': 1, 'H7': 1}), 'H8'))
        _check('막힌 3 + 열린 3 은 허용', not _forbidden(_board_from({'E8': -1, 'F8': 1, 'G8': 1, 'H6': 1, 'H7': 1}), 'H8'))
        _check('4-3 허용', not _forbidden(_board_from({'E8': 1, 'F8': 1, 'G8': 1, 'H6': 1, 'H7': 1}), 'H8'))
        _check('가장자리 막힌 3 은 3-3 아님', not _forbidden(_board_from({'A2': 1, 'A3': 1, 'B1': 1, 'C1': 1}), 'A1'))
        _check('5목 완성은 금수보다 우선',
               not _forbidden(_board_from({'D8': 1, 'E8': 1, 'F8': 1, 'G8': 1, 'H6': 1, 'H7': 1, 'J9': 1, 'K10': 1}), 'H8'))
        _check('합법수에서 금수 제외', not _board_from(B33).legal()[parse_move('H8')])
    W33 = {k: -1 for k in B33}
    _check('백 3-3: ' + ('금지 (한국식)' if RULE_MODE == 2 else '허용'), _forbidden(_board_from(W33), 'H8', -1) == (RULE_MODE == 2))
    if RULE_MODE == 3:
        # 렌주 전용
        B44 = {'E8': 1, 'F8': 1, 'G8': 1, 'H5': 1, 'H6': 1, 'H7': 1}      # H8: 가로 4 + 세로 4
        _check('렌주: 흑 4-4 금지', _forbidden(_board_from(B44), 'H8'))
        _check('렌주: 백 4-4 허용', not _forbidden(_board_from({k: -1 for k in B44}), 'H8', -1))
        _check('렌주: 한 줄 4-4 (●_●●●_●) 금지', _forbidden(_board_from({'D8': 1, 'F8': 1, 'H8': 1, 'K8': 1}), 'G8'))
        _check('렌주: 열린 4 하나 (_●●●●_) 는 허용', not _forbidden(_board_from({'E8': 1, 'F8': 1, 'G8': 1}), 'H8'))
        B6 = {'C8': 1, 'D8': 1, 'E8': 1, 'F8': 1, 'G8': 1}
        _check('렌주: 흑 6목은 금수', _forbidden(_board_from(B6), 'H8'))
        _check('렌주: 흑 6목은 승리 아님', not _five_after(B6, 'H8', 1))
        _check('렌주: 백 6목은 승리', _five_after({k: -1 for k in B6}, 'H8', -1))
        _check('렌주: 흑 정확히 5목은 승리', _five_after({'D8': 1, 'E8': 1, 'F8': 1, 'G8': 1}, 'H8', 1))
        _check('렌주: 5목이 되면 4-4여도 승리',
               not _forbidden(_board_from({'D8': 1, 'E8': 1, 'F8': 1, 'G8': 1, 'H5': 1, 'H6': 1, 'H7': 1}), 'H8'))
    else:
        _check('5목 승리', _five_after({'D8': 1, 'E8': 1, 'F8': 1, 'G8': 1}, 'H8', 1))
        _check('6목 판정 (exact_five=True면 무효)', _five_after({'C8': 1, 'D8': 1, 'E8': 1, 'F8': 1, 'G8': 1}, 'H8', 1) == (not EXACT))
    # 대각선 5목
    g = Gomoku()
    for mv in ['H8', 'A1', 'J9', 'A3', 'K10', 'A5', 'L11', 'A7', 'G7']:
        g.play(parse_move(mv))
    _check('대각선 5목으로 흑 승리', g.winner == 1)
    print('모든 규칙 검증 통과')
else:
    print('규칙 검증은 15x15 기준으로 작성됨 — 건너뜀')


## 4. 비교 상대
- **휴리스틱**: 한 수 앞 탐욕 (공격 점수 + 0.9 × 수비 점수). 연습 상대 겸 평가 상대
- **미니맥스**: 내 후보 8수 × 상대 응수 8수를 읽고 남은 위협으로 평가. **평가 전용** (학습에 안 씀)

In [ ]:
# ===== 4. 비교 상대: 랜덤 / 휴리스틱 / 미니맥스 =====
# 점수표: 5목 > 열린4 > 4 > 열린3 > 닫힌3/열린2 ...
S_FIVE, S_OPEN4, S_FOUR, S_OPEN3, S_CLOSED3, S_OPEN2, S_CLOSED2, S_ONE = 100000, 10000, 1500, 1000, 150, 100, 15, 2

@njit
def line_score(line, exact):
    if line_five_with_center(line, exact): return S_FIVE
    if line_open_four(line, exact): return S_OPEN4
    if line_is_four(line, exact): return S_FOUR
    if line_open_three(line, exact): return S_OPEN3
    # 연속 개수 + 열린 끝 개수로 나머지 평가
    k = 1; i = 4
    while i >= 0 and line[i] == 1: k += 1; i -= 1
    lo = i >= 0 and line[i] == 0
    j = 6
    while j <= 10 and line[j] == 1: k += 1; j += 1
    ro = j <= 10 and line[j] == 0
    opens = int(lo) + int(ro)
    if opens == 0: return 0
    if k >= 3: return S_CLOSED3
    if k == 2: return S_OPEN2 if opens == 2 else S_CLOSED2
    return S_ONE * opens

@njit
def point_score(board, r, c, color, n, exact):
    """(r,c)에 color가 둔다면 생기는 모양 점수"""
    board[r, c] = color
    s = 0
    for d in range(4):
        s += line_score(get_line(board, r, c, d, color, n), exact)
    board[r, c] = 0
    return s

@njit
def candidates(board, n):
    """돌 주변 2칸 안의 빈칸"""
    near = np.zeros((n, n), dtype=np.bool_)
    has = False
    for r in range(n):
        for c in range(n):
            if board[r, c] != 0:
                has = True
                for dr in range(-2, 3):
                    for dc in range(-2, 3):
                        rr = r + dr; cc = c + dc
                        if 0 <= rr < n and 0 <= cc < n and board[rr, cc] == 0:
                            near[rr, cc] = True
    if not has:
        near[n // 2, n // 2] = True
    return near

@njit
def heuristic_scores(board, color, legal, n, exact, exact_opp):
    """각 칸 점수 = 공격 점수 + 0.9 × 수비 점수 (불법/먼 칸은 -1). exact/exact_opp = 내/상대의 5목 판정 방식"""
    near = candidates(board, n)
    out = np.full(n * n, -1.0)
    for r in range(n):
        for c in range(n):
            a = r * n + c
            if legal[a] and near[r, c]:
                out[a] = point_score(board, r, c, color, n, exact) + 0.9 * point_score(board, r, c, -color, n, exact_opp)
    if out.max() < 0:            # 주변에 둘 곳이 없으면 아무 합법수
        for a in range(n * n):
            if legal[a]: out[a] = 0.0
    return out

def heuristic_move(g, noise=0.0, rng=np.random):
    s = heuristic_scores(g.board, g.player, g.legal(), N, EX(g.player), EX(-g.player))
    if noise > 0:
        s = np.where(s >= 0, s * (1 + noise * rng.random(NN)), s)
    best = np.flatnonzero(s == s.max())
    return int(rng.choice(best))

def random_move(g, rng=np.random):
    return int(rng.choice(np.flatnonzero(g.legal())))

@njit
def best_attack(board, color, n, exact):
    near = candidates(board, n)
    best = 0
    for r in range(n):
        for c in range(n):
            if near[r, c]:
                s = point_score(board, r, c, color, n, exact)
                if s > best: best = s
    return best

def leaf_value(board, me, forced_depth=1):
    """내 차례인 국면 평가
    - 내가 바로 5목 가능 → 승 / 상대 5목 자리 2곳 이상 → 패
    - 상대 5목 자리가 1곳이면 그 자리를 막은 뒤 상대 입장에서 다시 평가
    - 내가 열린4를 만들 수 있고 상대는 4를 못 만들면 → 거의 승
    - 그 외: 내 최고 공격 - 0.8 × 상대 최고 공격"""
    if best_attack(board, me, N, EX(me)) >= S_FIVE: return 1e8
    opp_five = winning_moves(board, -me, N, EX(-me))
    if len(opp_five) >= 2: return -1e8
    if len(opp_five) == 1:
        b = opp_five[0]; r, c = divmod(b, N)
        if is_forbidden(board, r, c, me, N, RULE_MODE, EX(me)):
            return -1e8                      # 막을 자리가 금수
        if forced_depth <= 0: return 0.0
        board[r, c] = me
        v = -leaf_value(board, -me, forced_depth - 1)
        board[r, c] = 0
        return v
    my = best_attack(board, me, N, EX(me))
    opp = best_attack(board, -me, N, EX(-me))
    if my >= S_OPEN4 and opp < S_FOUR: return 5e7
    return my - 0.8 * opp

def minimax_move(g, k=8, rng=np.random):
    """평가 전용: 내 후보 k개 × 상대 응수 k개를 읽고, 남은 위협(내 최고 공격 - 상대 최고 공격)으로 평가"""
    me = g.player
    wins = [a for a in winning_moves(g.board, me, N, EX(me)) if g.legal()[a]]
    if wins: return int(wins[0])
    s = heuristic_scores(g.board, me, g.legal(), N, EX(me), EX(-me))
    order = np.argsort(-s)[:k]
    order = [int(a) for a in order if s[a] >= 0]
    best_a, best_v = order[0], -1e18
    for a in order:
        g2 = g.copy().play(a)
        if g2.done():
            v = 1e9 if g2.winner == me else 0
        else:
            opp_wins = winning_moves(g2.board, -me, N, EX(-me))
            if len(opp_wins) > 0:
                v = -1e9
            else:
                s2 = heuristic_scores(g2.board, -me, g2.legal(), N, EX(-me), EX(me))
                replies = [int(b) for b in np.argsort(-s2)[:k] if s2[b] >= 0]
                v = 1e18
                for b in replies:
                    g3 = g2.copy().play(b)
                    if g3.done():
                        vb = -1e9 if g3.winner == -me else 0
                    else:
                        vb = leaf_value(g3.board, me)
                    v = min(v, vb)
                if not replies: v = 0
        v += rng.random() * 1e-3   # 동점이면 무작위
        if v > best_v: best_a, best_v = a, v
    return best_a

def play_bots(f_black, f_white, show_board=False):
    g = Gomoku()
    while not g.done():
        g.legal()
        if g.done(): break
        g.play((f_black if g.player == 1 else f_white)(g))
    if show_board: show(g)
    return g.winner

t = time.time()
_res = {'승': 0, '무': 0, '패': 0}
for i in range(6):
    side = 1 if i % 2 == 0 else -1
    hb = lambda g: heuristic_move(g, noise=0.3)
    w = play_bots(minimax_move, hb) if side == 1 else play_bots(hb, minimax_move)
    _res['승' if w == side else ('무' if w == 0 else '패')] += 1
print(f'미니맥스 vs 휴리스틱 시험 6판: {_res}  ({time.time()-t:.1f}s)')


## 5. 신경망
ResNet (64채널, 잔차 블록 6개). 입력 5채널: 내 돌 / 상대 돌 / 마지막 수 / 흑 차례 / 금수 자리

In [ ]:
# ===== 5. 신경망: ResNet + 정책 헤드 + 가치 헤드 =====
# 입력 5채널 (둘 차례 기준): 내 돌, 상대 돌, 마지막 수, 흑 차례 여부, 금수 자리
IN_CH = 5

def encode(g):
    x = np.zeros((IN_CH, N, N), dtype=np.int8)
    x[0] = g.board == g.player
    x[1] = g.board == -g.player
    if g.last >= 0:
        x[2].flat[g.last] = 1
    if g.player == 1:
        x[3] = 1
    if g.moves > 0:
        x[4] = ((g.board == 0).reshape(-1) & ~g.legal()).reshape(N, N)
    return x

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False); self.b1 = nn.BatchNorm2d(ch)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False); self.b2 = nn.BatchNorm2d(ch)
    def forward(self, x):
        y = F.relu(self.b1(self.c1(x)))
        y = self.b2(self.c2(y))
        return F.relu(x + y)

class Net(nn.Module):
    def __init__(self, ch=CFG['channels'], blocks=CFG['blocks']):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(IN_CH, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU())
        self.body = nn.Sequential(*[ResBlock(ch) for _ in range(blocks)])
        # 정책: 완전 합성곱 (칸마다 점수 하나)
        self.pol = nn.Sequential(nn.Conv2d(ch, 32, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(), nn.Conv2d(32, 1, 1))
        # 가치: 전역 평균 풀링 → 승패 예측 [-1, 1]
        self.val_conv = nn.Sequential(nn.Conv2d(ch, 32, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU())
        self.val_fc = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 1), nn.Tanh())
    def forward(self, x):
        h = self.body(self.stem(x))
        p = self.pol(h).flatten(1)                       # (B, N*N) 로짓
        v = self.val_fc(self.val_conv(h).mean((2, 3)))   # (B, 1)
        return p, v.squeeze(1)

@torch.no_grad()
def evaluate_batch(net, xs, masks):
    """xs: (B,C,N,N) int8, masks: (B,N*N) bool → 합법수 확률 (B,N*N), 가치 (B,)"""
    x = torch.from_numpy(np.asarray(xs)).to(DEVICE, non_blocking=True).float()
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
        logits, v = net(x)
    logits = logits.float()
    m = torch.from_numpy(np.asarray(masks)).to(DEVICE)
    logits = logits.masked_fill(~m, -1e9)
    p = torch.softmax(logits, 1).cpu().numpy()
    return p, v.float().cpu().numpy()

_net = Net().to(DEVICE)
print(f'파라미터 수: {sum(p.numel() for p in _net.parameters()) / 1e6:.2f}M')
del _net


## 6. MCTS
PUCT 탐색. 여러 판의 잎 노드를 모아 한 번에 GPU로 평가합니다.

In [ ]:
# ===== 6. MCTS (PUCT) — 여러 판의 잎 노드를 묶어 GPU 배치 평가 =====
def outcome_value(moves):
    """끝난 판의 승패 크기 (판 길이 반영). 승자 기준 +값, 패자 기준 -값으로 씀"""
    L = CFG['len_scale']
    return 1.0 - CFG['len_shaping'] * min(moves, L) / L

class Node:
    __slots__ = ('acts', 'P', 'Nv', 'W', 'children')
    def __init__(self):
        self.acts = None
    def expanded(self):
        return self.acts is not None
    def expand(self, probs, legal):
        acts = np.flatnonzero(legal)
        p = probs[acts].astype(np.float64)
        s = p.sum()
        self.acts = acts
        self.P = p / s if s > 0 else np.full(len(acts), 1.0 / len(acts))
        self.Nv = np.zeros(len(acts), dtype=np.float64)
        self.W = np.zeros(len(acts), dtype=np.float64)
        self.children = [None] * len(acts)
    def add_noise(self, alpha, eps):
        self.P = (1 - eps) * self.P + eps * np.random.dirichlet([alpha] * len(self.P))
    def select(self, c_puct):
        total = self.Nv.sum()
        q = np.where(self.Nv > 0, self.W / np.maximum(self.Nv, 1), 0.0)
        u = c_puct * self.P * (math.sqrt(total) if total > 0 else 1.0) / (1 + self.Nv)
        return int(np.argmax(q + u))

def _backup(path, v):
    """v: 잎 국면에서 '둘 차례'인 쪽 기준 가치. 위로 올라가며 부호를 뒤집음"""
    for node, i in reversed(path):
        v = -v
        node.Nv[i] += 1
        node.W[i] += v

def mcts_search(games, net, sims, noise=False):
    """games: 진행 중인 Gomoku 목록 → (각 판의 방문수 배열 (N*N), 루트 가치 추정)"""
    net.eval()
    roots = [Node() for _ in games]
    p, v0 = evaluate_batch(net, [encode(g) for g in games], [g.legal() for g in games])
    for r, g, pi in zip(roots, games, p):
        r.expand(pi, g.legal())
        if noise: r.add_noise(CFG['dirichlet_alpha'], CFG['dirichlet_eps'])
    c = CFG['c_puct']
    for _ in range(sims):
        pend = []
        for i, root in enumerate(roots):
            node, g, path = root, games[i].copy(), []
            while node.expanded():
                k = node.select(c)
                path.append((node, k))
                g.play(node.acts[k])
                if node.children[k] is None:
                    node.children[k] = Node()
                node = node.children[k]
                if g.done(): break
            if not g.done():
                g.legal()                     # 금수만 남았으면 여기서 무승부 처리됨
            if g.done():
                _backup(path, 0.0 if g.winner == 0 else -outcome_value(g.moves))   # 직전에 둔 쪽이 이김 → 둘 차례 기준 음수
            else:
                pend.append((node, g, path))
        if pend:
            p, v = evaluate_batch(net, [encode(g) for _, g, _ in pend], [g.legal() for _, g, _ in pend])
            for (node, g, path), pi, vi in zip(pend, p, v):
                node.expand(pi, g.legal())
                _backup(path, float(vi))
    visits = np.zeros((len(games), NN), dtype=np.float32)
    for j, r in enumerate(roots):
        visits[j, r.acts] = r.Nv
    return visits, v0

def tactical_move(g):
    """이길 수 있으면 그 수, 상대의 5목 자리가 하나면 막는 수, 아니면 None"""
    legal = g.legal()
    wins = [a for a in winning_moves(g.board, g.player, N, EX(g.player)) if legal[a]]
    if wins: return int(wins[0])
    threats = winning_moves(g.board, -g.player, N, EX(-g.player))
    if len(threats) == 1 and legal[threats[0]]:
        return int(threats[0])
    return None

def pick(visits, temperature_on):
    if temperature_on:
        return int(np.random.choice(NN, p=visits / visits.sum()))
    best = np.flatnonzero(visits == visits.max())
    return int(np.random.choice(best))

def ai_move(net, g, sims=None, verbose=False):
    """사람과 둘 때 쓰는 한 수 (잡음·무작위 없음)"""
    sims = sims or CFG['play_sims']
    if g.legal().sum() == 1:
        return int(np.flatnonzero(g.legal())[0])
    if CFG['tactical_override']:
        t = tactical_move(g)
        if t is not None: return t
    visits, v = mcts_search([g], net, sims)
    a = pick(visits[0], False)
    if verbose:
        print(f'AI 착수 {move_name(a)} | 승리 예상 {(v[0] + 1) / 2 * 100:.0f}% | 탐색 {sims}회')
    return a


## 7. 대국 실행기 & 리플레이 버퍼

In [ ]:
# ===== 7. 대국 실행기 (자기대전 / 아레나 / 평가 공용) & 리플레이 버퍼 =====
# 선수 표현: ('net', 신경망, 잡음여부, 무작위수) 또는 ('bot', 함수)
def run_games(black_of, white_of, n_games, sims, record=False):
    """black_of(i), white_of(i): i번째 판의 흑/백 선수.
    record=True면 신경망 쪽 국면을 (입력, 탐색분포, 둔 쪽) 으로 기록.
    반환: winners 리스트, records 리스트(판별)"""
    games = [Gomoku() for _ in range(n_games)]
    recs = [[] for _ in range(n_games)]
    while True:
        active = []
        for i, g in enumerate(games):
            if not g.done():
                g.legal()                      # 금수만 남은 경우 무승부 처리
                if not g.done(): active.append(i)
        if not active: break
        groups = {}                            # 신경망별로 묶어서 한 번에 탐색
        for i in active:
            g = games[i]
            pl = black_of(i) if g.player == 1 else white_of(i)
            legal = g.legal()
            if legal.sum() == 1:               # 둘 곳이 하나뿐 (흑 첫 수 천원 등)
                g.play(int(np.flatnonzero(legal)[0])); continue
            if pl[0] == 'bot':
                g.play(pl[1](g)); continue
            if CFG['tactical_override']:
                t = tactical_move(g)
                if t is not None:
                    if record:
                        pi = np.zeros(NN, dtype=np.float32); pi[t] = 1
                        recs[i].append((encode(g), pi, g.player))
                    g.play(t); continue
            key = (id(pl[1]), pl[2], pl[3])
            groups.setdefault(key, (pl, []))[1].append(i)
        for (pl, idx) in groups.values():
            _, net, noise, temp_moves = pl
            visits, _ = mcts_search([games[i] for i in idx], net, sims, noise=noise)
            for j, i in enumerate(idx):
                g = games[i]
                if record:
                    recs[i].append((encode(g), visits[j] / visits[j].sum(), g.player))
                g.play(pick(visits[j], g.moves < temp_moves))
    return [g.winner for g in games], recs, [g.moves for g in games]

def self_play(best):
    G = CFG['games_per_iter']
    n_h = int(round(G * CFG['heuristic_frac']))
    n_s = G - n_h
    hbot = ('bot', lambda g: heuristic_move(g, noise=0.2))
    ai_self = ('net', best, True, CFG['temp_moves'])
    ai_h = ('net', best, CFG['heuristic_noise'], CFG['heuristic_temp_moves'])
    # 0..n_s-1: 순수 자기대전, n_s..: 휴리스틱 판 (AI 흑백 교대)
    black_of = lambda i: ai_self if i < n_s else (ai_h if (i - n_s) % 2 == 0 else hbot)
    white_of = lambda i: ai_self if i < n_s else (hbot if (i - n_s) % 2 == 0 else ai_h)
    winners, recs, lens = run_games(black_of, white_of, G, CFG['sims'], record=True)
    samples = []
    for w, rec, ln in zip(winners, recs, lens):
        mag = outcome_value(ln)                     # 판 길이를 반영한 승패 크기
        for x, pi, pl in rec:
            samples.append((x, pi.astype(np.float16), float(w * pl * mag)))   # 무승부 0
    st = {}
    sw = winners[:n_s]
    st['self_black'] = sum(w == 1 for w in sw) / max(1, n_s)
    st['self_white'] = sum(w == -1 for w in sw) / max(1, n_s)
    hw = [(w == (1 if k % 2 == 0 else -1)) + 0.5 * (w == 0) for k, w in enumerate(winners[n_s:])]
    st['sp_vs_heur'] = float(np.mean(hw)) if hw else float('nan')
    st['avg_len'] = float(np.mean(lens))
    return samples, st

class Buffer:
    def __init__(self, cap):
        self.cap = cap
        self.x = np.zeros((cap, IN_CH, N, N), dtype=np.int8)
        self.pi = np.zeros((cap, NN), dtype=np.float16)
        self.z = np.zeros(cap, dtype=np.float32)
        self.n = 0; self.ptr = 0
    def add(self, samples):
        for x, pi, z in samples:
            self.x[self.ptr] = x; self.pi[self.ptr] = pi; self.z[self.ptr] = z
            self.ptr = (self.ptr + 1) % self.cap; self.n = min(self.n + 1, self.cap)
    def sample(self, bs):
        idx = np.random.randint(0, self.n, bs)
        x = self.x[idx].copy(); pi = self.pi[idx].astype(np.float32).reshape(bs, N, N); z = self.z[idx]
        # 8가지 회전·반사 대칭 무작위 적용 (오목 규칙은 대칭에 대해 보존됨)
        for b in range(bs):
            k = np.random.randint(8)
            if k:
                tx = np.rot90(x[b], k % 4, axes=(1, 2)); tp = np.rot90(pi[b], k % 4)
                if k >= 4:
                    tx = tx[:, :, ::-1]; tp = tp[:, ::-1]
                x[b] = tx; pi[b] = tp
        return x, pi.reshape(bs, NN), z
    def save(self, path):
        np.savez_compressed(path, x=self.x[:self.n], pi=self.pi[:self.n], z=self.z[:self.n], ptr=self.ptr)
    def load(self, path):
        d = np.load(path)
        k = min(len(d['z']), self.cap)
        self.x[:k] = d['x'][:k]; self.pi[:k] = d['pi'][:k]; self.z[:k] = d['z'][:k]
        self.n = k; self.ptr = int(d['ptr']) % self.cap if k == self.cap else k


## 8. 학습 / 아레나 / 평가 / 체크포인트

In [ ]:
# ===== 8. 학습 / 아레나 / 평가 / 체크포인트 =====
def train(net, opt, buf):
    net.train()
    tot_p = tot_v = 0.0
    for _ in range(CFG['train_steps']):
        x, pi, z = buf.sample(CFG['batch_size'])
        x = torch.from_numpy(x).to(DEVICE).float()
        pi = torch.from_numpy(pi).to(DEVICE)
        z = torch.from_numpy(z).to(DEVICE)
        logits, v = net(x)
        lp = -(pi * F.log_softmax(logits, 1)).sum(1).mean()   # 정책: 탐색 분포 따라 하기
        lv = F.mse_loss(v, z)                                  # 가치: 승패 예측
        opt.zero_grad(); (lp + lv).backward(); opt.step()
        tot_p += lp.item(); tot_v += lv.item()
    k = CFG['train_steps']
    return tot_p / k, tot_v / k

def match(net_a, opp, n, sims=None):
    """net_a vs opp (신경망 또는 봇 함수), 흑백 교대. 반환: 점수(승1 무0.5), 흑일 때 점수, 백일 때 점수, 무승부 수"""
    sims = sims or CFG['sims']
    A = ('net', net_a, False, CFG['arena_temp_moves'])
    B = ('net', opp, False, CFG['arena_temp_moves']) if isinstance(opp, nn.Module) else ('bot', opp)
    winners, _, _ = run_games(lambda i: A if i % 2 == 0 else B, lambda i: B if i % 2 == 0 else A, n, sims)
    sc = [(w == (1 if i % 2 == 0 else -1)) + 0.5 * (w == 0) for i, w in enumerate(winners)]
    sb, sw = sc[0::2], sc[1::2]
    return float(np.mean(sc)), float(np.mean(sb)), float(np.mean(sw)), int(sum(w == 0 for w in winners))

def get_lr(it):
    lr = None
    for k in sorted(CFG['lr_schedule']):
        if it >= k: lr = CFG['lr_schedule'][k]
    return lr

P_BEST, P_CUR, P_BUF, P_HIST = [os.path.join(SAVE_DIR, f) for f in ('best_model.pt', 'current.pt', 'buffer.npz', 'history.json')]
P_VFLAG = os.path.join(SAVE_DIR, 'value_target_v2.flag')   # 판 길이 반영 가치 목표로 바뀐 뒤인지 표시

def save_all(it, best, cur, opt, buf, hist):
    torch.save(best.state_dict(), P_BEST + '.tmp'); os.replace(P_BEST + '.tmp', P_BEST)
    torch.save({'it': it, 'net': cur.state_dict(), 'opt': opt.state_dict()}, P_CUR + '.tmp'); os.replace(P_CUR + '.tmp', P_CUR)
    buf.save(P_BUF.replace('.npz', '_tmp.npz')); os.replace(P_BUF.replace('.npz', '_tmp.npz'), P_BUF)
    with open(P_HIST, 'w') as f: json.dump(hist, f)
    if CFG['len_shaping'] > 0 and not os.path.exists(P_VFLAG):
        open(P_VFLAG, 'w').write('len_shaping')

def load_all():
    best, cur = Net().to(DEVICE), Net().to(DEVICE)
    opt = torch.optim.AdamW(cur.parameters(), lr=get_lr(0), weight_decay=CFG['weight_decay'])
    buf = Buffer(CFG['buffer_size']); hist = []; start = 0
    if os.path.exists(P_CUR):
        ck = torch.load(P_CUR, map_location=DEVICE)
        cur.load_state_dict(ck['net']); opt.load_state_dict(ck['opt']); start = ck['it'] + 1
        best.load_state_dict(torch.load(P_BEST, map_location=DEVICE))
        if os.path.exists(P_BUF): buf.load(P_BUF)
        if os.path.exists(P_HIST): hist = json.load(open(P_HIST))
        if CFG['len_shaping'] > 0 and not os.path.exists(P_VFLAG) and buf.n > 0:
            print(f'가치 목표가 바뀌어(판 길이 반영) 예전 방식의 버퍼 {buf.n:,} 국면은 비우고 시작합니다 (모델은 그대로 이어서 학습)')
            buf.n = 0; buf.ptr = 0
        print(f'체크포인트에서 이어서 학습: 반복 {start}부터 (버퍼 {buf.n:,} 국면)')
    else:
        best.load_state_dict(cur.state_dict())
        print('처음부터 학습 시작')
    return start, best, cur, opt, buf, hist

# 대국용: 저장된 최고 모델 불러오기 (11번·12번 셀에서 사용)
def load_play_net():
    net = Net().to(DEVICE)
    if os.path.exists(P_BEST):
        net.load_state_dict(torch.load(P_BEST, map_location=DEVICE)); print('best_model.pt 불러옴')
    else:
        print('⚠️ 저장된 모델이 없어 무작위 신경망으로 둡니다')
    net.eval()
    return net


## 9. 학습 루프
반복당 대략 5~8분 (T4 기준, 판 길이에 따라 다름). 출력 해석:
- **아레나**: 학습 모델 vs 최고 모델 20판, 55% 이상이면 교체
- **연습판 vs휴리스틱**: 자기대전 중 휴리스틱 판 성적
- **평가**: 3번마다 최고 모델 vs 휴리스틱 / 미니맥스 각 20판 (무승부 0.5점). 오목은 흑백 유불리가 커서 흑·백 성적을 따로 표시

In [ ]:
# ===== 9. 학습 루프 =====
# ① 자기대전 → ② 리플레이 버퍼 → ③ 학습 → ④ 아레나 → ⑤ 교체 여부 → (3번마다) 휴리스틱·미니맥스 평가
# 세션이 끊겨도 이 셀까지 다시 실행하면 저장된 체크포인트에서 이어서 학습합니다.
start, best, cur, opt, buf, hist = load_all()

for it in range(start, CFG['iterations']):
    t0 = time.time()
    lr = get_lr(it)
    for gp in opt.param_groups: gp['lr'] = lr      # 저장된 옵티마이저의 예전 학습률을 덮어씀

    samples, st = self_play(best)
    buf.add(samples)
    t1 = time.time()

    lp = lv = float('nan')
    if buf.n >= CFG['batch_size'] * 8:
        lp, lv = train(cur, opt, buf)
    t2 = time.time()

    score, sb, sw, draws = match(cur, best, CFG['arena_games'])
    replaced = score >= CFG['arena_threshold']
    if replaced:
        best.load_state_dict(cur.state_dict())
    t3 = time.time()

    rec = dict(it=it, lr=lr, n_samples=len(samples), buffer=buf.n, loss_p=lp, loss_v=lv,
               arena=score, arena_draws=draws, replaced=replaced, **st)
    line = (f'[{it:3d}] 국면+{len(samples)} 버퍼 {buf.n:,} | 손실 정책 {lp:.3f} 가치 {lv:.3f} | '
            f'아레나 {score*100:.0f}% (무 {draws}) {"→ 교체 ✅" if replaced else "유지"} | '
            f'자기대전 흑승 {st["self_black"]*100:.0f}% 백승 {st["self_white"]*100:.0f}% 평균 {st["avg_len"]:.0f}수 | '
            f'연습판 vs휴리스틱 {st["sp_vs_heur"]*100:.0f}%')

    if it % CFG['eval_every'] == 0:
        h, hb, hw, _ = match(best, lambda g: heuristic_move(g), CFG['eval_games'])
        m, mb, mw, _ = match(best, minimax_move, CFG['eval_games'])
        rec.update(vs_heur=h, vs_heur_b=hb, vs_heur_w=hw, vs_minimax=m, vs_minimax_b=mb, vs_minimax_w=mw)
        line += (f'\n      평가 → vs휴리스틱 {h*100:.0f}% (흑 {hb*100:.0f}/백 {hw*100:.0f}) | '
                 f'vs미니맥스 {m*100:.0f}% (흑 {mb*100:.0f}/백 {mw*100:.0f})')
    t4 = time.time()
    rec['sec'] = t4 - t0
    hist.append(rec)
    save_all(it, best, cur, opt, buf, hist)
    print(line + f'\n      시간: 자기대전 {t1-t0:.0f}s 학습 {t2-t1:.0f}s 아레나 {t3-t2:.0f}s 평가 {t4-t3:.0f}s', flush=True)


## 10. 학습 그래프

In [ ]:
# ===== 10. 학습 그래프 =====
import matplotlib.pyplot as plt
hist = json.load(open(P_HIST)) if os.path.exists(P_HIST) else []
if hist:
    it = [h['it'] for h in hist]
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    ax[0].plot(it, [h['loss_p'] for h in hist], label='policy loss')
    ax[0].plot(it, [h['loss_v'] for h in hist], label='value loss')
    ax[0].set_title('Loss'); ax[0].legend(); ax[0].set_xlabel('iteration')
    ax[1].plot(it, [h['arena'] for h in hist], '.', alpha=.5, label='arena (cand vs best)')
    rep = [h['it'] for h in hist if h['replaced']]
    ax[1].vlines(rep, 0, 1, colors='g', alpha=.15, label='best replaced')
    ax[1].plot(it, [h['sp_vs_heur'] for h in hist], alpha=.7, label='self-play vs heuristic')
    ax[1].set_ylim(0, 1); ax[1].set_title('Arena / practice games'); ax[1].legend(); ax[1].set_xlabel('iteration')
    ev = [h for h in hist if 'vs_heur' in h]
    ax[2].plot([h['it'] for h in ev], [h['vs_heur'] for h in ev], 'o-', label='vs heuristic')
    ax[2].plot([h['it'] for h in ev], [h['vs_minimax'] for h in ev], 's-', label='vs minimax (eval only)')
    ax[2].set_ylim(0, 1); ax[2].set_title('Evaluation (score, draw = 0.5)'); ax[2].legend(); ax[2].set_xlabel('iteration')
    plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'training.png'), dpi=110); plt.show()
else:
    print('아직 학습 기록이 없습니다')


## 11. AI와 대국 — 좌표 입력
맨 아래 줄의 `#`을 지우고 실행하세요.

In [ ]:
# ===== 11. AI와 대국 (좌표 입력 방식, Colab 밖에서도 동작) =====
# 1, 2, 4, 5, 6, 7, 8번 셀 실행 후 사용 (학습 루프 9번은 안 돌려도 됨). 좌표 예: H8 (가로 A~P, I 제외 / 세로 1~15), q = 종료
def play_text(human=1, sims=None):
    net = load_play_net()
    g = Gomoku()
    print('당신은', '흑(●, 선수)' if human == 1 else '백(○, 후수)')
    while not g.done():
        g.legal()
        if g.done(): break
        if g.player == human:
            show(g)
            s = input('착수 (예: H8): ')
            if s.strip().lower() == 'q': return
            try:
                a = parse_move(s)
            except Exception:
                print('좌표 형식이 잘못됐습니다'); continue
            if not g.legal()[a]:
                print('둘 수 없는 자리입니다 (이미 돌이 있거나 금수, 또는 흑 첫 수는 천원)'); continue
            g.play(a)
        else:
            g.play(ai_move(net, g, sims, verbose=True))
    show(g)
    print({1: '흑 승리', -1: '백 승리', 0: '무승부'}[g.winner], '— 당신의', '승리!' if g.winner == human else ('무승부' if g.winner == 0 else '패배'))

# play_text(human=1)    # 흑으로 두기
# play_text(human=-1)   # 백으로 두기


## 12. AI와 대국 — 판 클릭 (Colab)
`새 게임` 버튼으로 흑/백 선택. `×` 표시는 흑의 금수 자리(3-3, 4-4, 6목)입니다.

In [ ]:
# ===== 12. AI와 대국 (판을 클릭해서 두기, Colab용) =====
import ipywidgets as W
from IPython.display import display, HTML

CELL = 32                      # 칸 크기(px)
_L = (N - 1) * CELL            # 줄이 그려지는 영역 크기
_STONE_B = 'radial-gradient(circle at 35% 30%, #8a8a8a 0, #2a2a2a 40%, #000 72%)'
_STONE_W = 'radial-gradient(circle at 35% 30%, #ffffff 0, #f0f0f0 45%, #b9b9b9 85%)'
_LAST = 'radial-gradient(circle, #e53935 0 4px, transparent 4.5px)'
BOARD_CSS = f'''
<style>
.gm-wood {{ background: linear-gradient(135deg, #e3bd6b, #d4a94f); padding: 14px; border-radius: 6px;
           box-shadow: 0 4px 12px rgba(0,0,0,.35); width: max-content; }}
.gm-grid {{ position: relative; }}
.gm-grid::before {{ content: ''; position: absolute; left: {CELL//2}px; top: {CELL//2}px; width: {_L}px; height: {_L}px;
           background-image: linear-gradient(90deg, #4a3a1a 1px, transparent 1px), linear-gradient(#4a3a1a 1px, transparent 1px);
           background-size: {CELL}px {CELL}px; border-right: 1px solid #4a3a1a; border-bottom: 1px solid #4a3a1a;
           pointer-events: none; }}
.gm-cell {{ position: relative; z-index: 1; width: {CELL}px !important; height: {CELL}px !important; min-width: 0 !important;
           margin: 0 !important; padding: 0 !important; border: none !important; border-radius: 50% !important;
           background: transparent !important; box-shadow: none !important; outline: none !important;
           color: #b03020 !important; font-weight: bold; font-size: 16px; line-height: {CELL}px; cursor: pointer; }}
.gm-cell.gm-star {{ background: radial-gradient(circle, #4a3a1a 0 3px, transparent 3.5px) !important; }}
.gm-cell:hover:not(.gm-black):not(.gm-white) {{ background: radial-gradient(circle, rgba(0,0,0,.28) 0 13px, transparent 14px) !important; }}
.gm-cell.gm-black {{ background: {_STONE_B} !important; box-shadow: 1px 2px 3px rgba(0,0,0,.6) !important; transform: scale(.92); }}
.gm-cell.gm-white {{ background: {_STONE_W} !important; box-shadow: 1px 2px 3px rgba(0,0,0,.45) !important; transform: scale(.92); }}
.gm-cell.gm-black.gm-last {{ background: {_LAST}, {_STONE_B} !important; }}
.gm-cell.gm-white.gm-last {{ background: {_LAST}, {_STONE_W} !important; }}
.gm-cell.gm-forbid::after {{ content: '×'; position: absolute; left: 0; top: 0; width: 100%; height: 100%; line-height: {CELL}px;
           text-align: center; color: #b03020; font-weight: bold; font-size: 18px; }}
</style>
'''
STARS = {(3, 3), (3, 11), (7, 7), (11, 3), (11, 11)} if N == 15 else set()   # 화점

class ClickBoard:
    def __init__(self, sims=None):
        self.net = load_play_net()
        self.sims = sims
        self.btn, self.cls = [], [set() for _ in range(NN)]
        for k in range(NN):
            b = W.Button(description='', layout=W.Layout(width=f'{CELL}px', height=f'{CELL}px', margin='0', padding='0'))
            b.add_class('gm-cell')
            if divmod(k, N) in STARS: b.add_class('gm-star')
            b.on_click(lambda _, k=k: self.click(k))
            self.btn.append(b)
        grid = W.GridBox(self.btn, layout=W.Layout(grid_template_columns=f'repeat({N}, {CELL}px)', grid_gap='0px'))
        grid.add_class('gm-grid')
        wood = W.Box([grid]); wood.add_class('gm-wood')
        self.status = W.HTML()
        nb = W.Button(description='새 게임: 흑(선수)'); nb.on_click(lambda _: self.new(1))
        nw = W.Button(description='새 게임: 백(후수)'); nw.on_click(lambda _: self.new(-1))
        display(HTML(BOARD_CSS))
        display(W.VBox([W.HBox([nb, nw]), self.status, wood]))
        self.new(1)

    def new(self, human):
        self.g = Gomoku(); self.human = human; self.busy = False
        self.msg('당신은 ' + ('흑 ⚫ (선수)' if human == 1 else '백 ⚪ (후수)') + ' · 빨간 점 = 마지막 수')
        self.draw()
        if human == -1: self.ai_turn()

    def msg(self, s): self.status.value = f'<b style="font-size:15px">{s}</b>'

    def draw(self):
        legal = self.g.legal() if not self.g.done() else np.zeros(NN, bool)
        my_turn = self.g.player == self.human and not self.g.done() and self.g.moves > 0
        for k, b in enumerate(self.btn):
            v = int(self.g.board.flat[k])
            want = {'gm-black'} if v == 1 else ({'gm-white'} if v == -1 else set())
            if k == self.g.last: want.add('gm-last')
            if v == 0 and my_turn and not legal[k]: want.add('gm-forbid')   # × = 금수
            for c in self.cls[k] - want: b.remove_class(c)
            for c in want - self.cls[k]: b.add_class(c)
            self.cls[k] = want

    def end_check(self):
        self.g.legal()
        if self.g.done():
            w = self.g.winner
            self.msg('무승부' if w == 0 else ('당신의 승리! 🎉' if w == self.human else 'AI 승리'))
            self.draw(); return True
        return False

    def click(self, k):
        if self.busy or self.g.done() or self.g.player != self.human: return
        if not self.g.legal()[k]:
            self.msg('둘 수 없는 자리입니다 (× = 금수 / 흑 첫 수는 정중앙)'); return
        self.g.play(k); self.draw()
        if not self.end_check(): self.ai_turn()

    def ai_turn(self):
        self.busy = True; self.msg('AI 생각 중...')
        a = ai_move(self.net, self.g, self.sims)
        self.g.play(a); self.busy = False
        self.draw()
        if not self.end_check():
            self.msg(f'AI: {move_name(a)} — 당신 차례')

board_ui = ClickBoard()


## 13. 자기대전 들여다보기
최고 모델끼리 한 판을 두고 착수 순서를 번호로 보여줍니다(홀수 = 흑, 짝수 = 백). 승패가 **이중 위협**(5목 자리가 두 곳 이상)으로 갈렸는지, **금수 유도**(흑이 막을 자리가 금수)로 갈렸는지도 알려줍니다.
- `analyze_selfplay(10)`: 10판의 흑백 승률과 패배 원인 통계
- 1, 2, 4~8번 셀 실행 후 사용 (9번 학습 셀은 건너뛰기)

In [ ]:
# ===== 13. 자기대전 들여다보기: 한 판 보기 & 패배 원인 통계 =====
# 1, 2, 4~8번 셀 실행 후 사용 (9번 학습 셀은 안 돌려도 됨)
def _selfplay_game(net, sims, temp_moves=4):
    """최고 모델끼리 한 판. 반환: (판, 착수 순서, 사건 메모 목록, 패배 원인)"""
    g = Gomoku(); order, notes = [], []
    cause = None
    while True:
        legal = g.legal()
        if g.done(): break
        me, opp = g.player, -g.player
        name = '흑' if me == 1 else '백'
        oname = '백' if me == 1 else '흑'
        threats = winning_moves(g.board, opp, N, EX(opp))          # 상대가 다음 수에 5목을 만들 자리
        my_wins = [a for a in winning_moves(g.board, me, N, EX(me)) if legal[a]]
        if threats and not my_wins:
            blocked = [t for t in threats if not legal[t]]
            if len(threats) >= 2:
                notes.append(f'{g.moves + 1}수째 ({name} 차례): {oname}의 5목 자리가 {len(threats)}곳 ({", ".join(move_name(t) for t in threats)}) → {name}이 다 막을 수 없음')
                cause = cause or ('이중 위협' if not blocked else '이중 위협(금수 포함)')
            elif blocked:
                notes.append(f'{g.moves + 1}수째 ({name} 차례): {oname}의 5목 자리 {move_name(blocked[0])}가 {name}의 금수라 막을 수 없음')
                cause = cause or '금수 유도'
        if legal.sum() == 1:
            a = int(np.flatnonzero(legal)[0])
        else:
            a = tactical_move(g)
            if a is None:
                visits, _ = mcts_search([g], net, sims)
                a = pick(visits[0], g.moves < temp_moves)
        order.append(a); g.play(a)
    if cause is None and g.winner != 0:
        cause = '기타'
    return g, order, notes, cause

def show_numbered(g, order):
    """착수 순서 번호로 판 출력. 홀수 = 흑, 짝수 = 백"""
    num = {a: i + 1 for i, a in enumerate(order)}
    print('    ' + ''.join(f'{c:>3}' for c in COLS))
    for r in range(N):
        row = ''
        for c in range(N):
            a = r * N + c
            row += f'{num[a]:>3}' if a in num else '  ·'
        print(f'{N - r:3d} ' + row)

def watch_selfplay(sims=None, temp_moves=4):
    """최고 모델끼리 한 판을 두고, 착수 순서와 승패가 갈린 장면을 보여줌"""
    net = load_play_net()
    g, order, notes, cause = _selfplay_game(net, sims or CFG['sims'], temp_moves)
    show_numbered(g, order)
    res = {1: '흑 승', -1: '백 승', 0: '무승부'}[g.winner]
    print(f'\n결과: {res} ({len(order)}수) | 홀수 번호 = 흑, 짝수 번호 = 백')
    print('착수 순서:', ' '.join(f'{i + 1}.{move_name(a)}' for i, a in enumerate(order)))
    for n in notes: print(' -', n)
    if cause: print('승패가 갈린 이유:', cause)

def analyze_selfplay(n=10, sims=None, temp_moves=4):
    """최고 모델끼리 n판을 두고 흑백 승률과 패배 원인을 집계 (판당 수십 초)"""
    net = load_play_net()
    stats = {}; lens = []; wins = {1: 0, -1: 0, 0: 0}
    t = time.time()
    for i in range(n):
        g, order, notes, cause = _selfplay_game(net, sims or CFG['sims'], temp_moves)
        wins[g.winner] += 1; lens.append(len(order))
        if g.winner != 0:
            key = ('흑 승' if g.winner == 1 else '백 승') + f' / {cause}'
            stats[key] = stats.get(key, 0) + 1
    print(f'{n}판 ({time.time() - t:.0f}s) | 흑승 {wins[1]} 백승 {wins[-1]} 무 {wins[0]} | 평균 {np.mean(lens):.0f}수')
    for k, v in sorted(stats.items(), key=lambda x: -x[1]):
        print(f'  {k}: {v}판')
    print('  (이중 위협 = 5목 자리가 두 곳 이상, 금수 유도 = 흑이 막을 자리가 금수, 기타 = 위의 경우 없이 끝남)')

watch_selfplay()
# analyze_selfplay(10)   # 여러 판의 패배 원인 통계
